# Hybrid ARIMA-LSTM AQI Forecasting Model

This is a standalone, all-in-one Jupyter Notebook that contains the complete pipeline for forecasting Air Quality Index (AQI) around Indian Oil Corporation refinery zones using a hybrid ARIMA-LSTM approach.

### Pipeline Phases:
1. **Configuration & Setup**
2. **Synthetic Data Generation**
3. **Data Preprocessing**
4. **Exploratory Data Analysis (EDA)**
5. **Feature Selection**
6. **Seasonal Decomposition**
7. **ARIMA Modeling**
8. **LSTM Modeling**
9. **Hybrid Forecast Combination**

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display


## 1. Configuration & Utility Functions

In [ ]:
"""
config.py — Global Configuration for AQI Forecasting Pipeline
==============================================================
Central configuration for the Hybrid ARIMA-LSTM AQI Forecasting Model.
All tunable parameters, file paths, and constants are defined here.
"""

import os

# ──────────────────────────────────────────────────────────────
# DIRECTORY PATHS
# ──────────────────────────────────────────────────────────────
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(BASE_DIR, "data")
PLOTS_DIR = os.path.join(BASE_DIR, "plots")
MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

# Sub-directories for plots
EDA_PLOTS_DIR = os.path.join(PLOTS_DIR, "eda")
DECOMP_PLOTS_DIR = os.path.join(PLOTS_DIR, "decomposition")
MODEL_PLOTS_DIR = os.path.join(PLOTS_DIR, "models")

# Create all directories
for d in [DATA_DIR, EDA_PLOTS_DIR, DECOMP_PLOTS_DIR, MODEL_PLOTS_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ──────────────────────────────────────────────────────────────
# DATA GENERATION SETTINGS
# ──────────────────────────────────────────────────────────────
DATE_RANGE_START = "2019-01-01"
DATE_RANGE_END = "2024-12-31"
MISSING_FRACTION = 0.15  # ~15% random missingness

# Refinery zones with baseline pollutant profiles (µg/m³ unless noted)
# Format: {pollutant: (mean, std_dev)}
REFINERY_ZONES = {
    "Mathura_Refinery": {
        "PM2.5": (85, 35),
        "PM10": (160, 55),
        "SO2": (45, 18),
        "NO2": (55, 22),
        "CO": (2.5, 1.0),       # mg/m³
        "O3": (35, 15),
        "NH3": (30, 12),
        "Pb": (0.3, 0.15),
        "Temperature": (28, 8),  # °C
        "Humidity": (60, 18),    # %
        "Wind_Speed": (8, 3.5),  # km/h
    },
    "Panipat_Refinery": {
        "PM2.5": (75, 30),
        "PM10": (145, 50),
        "SO2": (50, 20),
        "NO2": (48, 20),
        "CO": (2.2, 0.9),
        "O3": (40, 16),
        "NH3": (25, 10),
        "Pb": (0.25, 0.12),
        "Temperature": (26, 9),
        "Humidity": (55, 20),
        "Wind_Speed": (9, 4),
    },
    "Gujarat_Refinery": {
        "PM2.5": (65, 25),
        "PM10": (130, 45),
        "SO2": (55, 22),
        "NO2": (42, 18),
        "CO": (1.8, 0.8),
        "O3": (45, 18),
        "NH3": (22, 9),
        "Pb": (0.2, 0.1),
        "Temperature": (32, 6),
        "Humidity": (65, 15),
        "Wind_Speed": (12, 5),
    },
}

# Pollutants used for AQI sub-index calculation
AQI_POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "NH3", "Pb"]

# Meteorological features (not used for AQI calc, but as predictive features)
METEO_FEATURES = ["Temperature", "Humidity", "Wind_Speed"]

# All feature columns
ALL_FEATURES = AQI_POLLUTANTS + METEO_FEATURES

# ──────────────────────────────────────────────────────────────
# AQI BREAKPOINT TABLE (CPCB India Standard)
# Format: list of (BPLo, BPHi, ILo, IHi)
# ──────────────────────────────────────────────────────────────
AQI_BREAKPOINTS = {
    "PM2.5": [  # 24-hr avg, µg/m³
        (0, 30, 0, 50),
        (31, 60, 51, 100),
        (61, 90, 101, 200),
        (91, 120, 201, 300),
        (121, 250, 301, 400),
        (251, 500, 401, 500),
    ],
    "PM10": [  # 24-hr avg, µg/m³
        (0, 50, 0, 50),
        (51, 100, 51, 100),
        (101, 250, 101, 200),
        (251, 350, 201, 300),
        (351, 430, 301, 400),
        (431, 600, 401, 500),
    ],
    "SO2": [  # 24-hr avg, µg/m³
        (0, 40, 0, 50),
        (41, 80, 51, 100),
        (81, 380, 101, 200),
        (381, 800, 201, 300),
        (801, 1600, 301, 400),
        (1601, 2400, 401, 500),
    ],
    "NO2": [  # 24-hr avg, µg/m³
        (0, 40, 0, 50),
        (41, 80, 51, 100),
        (81, 180, 101, 200),
        (181, 280, 201, 300),
        (281, 400, 301, 400),
        (401, 600, 401, 500),
    ],
    "CO": [  # 8-hr avg, mg/m³
        (0, 1.0, 0, 50),
        (1.1, 2.0, 51, 100),
        (2.1, 10.0, 101, 200),
        (10.1, 17.0, 201, 300),
        (17.1, 34.0, 301, 400),
        (34.1, 50.0, 401, 500),
    ],
    "O3": [  # 8-hr avg, µg/m³
        (0, 50, 0, 50),
        (51, 100, 51, 100),
        (101, 168, 101, 200),
        (169, 208, 201, 300),
        (209, 748, 301, 400),
        (749, 1000, 401, 500),
    ],
    "NH3": [  # 24-hr avg, µg/m³
        (0, 200, 0, 50),
        (201, 400, 51, 100),
        (401, 800, 101, 200),
        (801, 1200, 201, 300),
        (1201, 1800, 301, 400),
        (1801, 2400, 401, 500),
    ],
    "Pb": [  # 24-hr avg, µg/m³
        (0, 0.5, 0, 50),
        (0.51, 1.0, 51, 100),
        (1.1, 2.0, 101, 200),
        (2.1, 3.0, 201, 300),
        (3.1, 3.5, 301, 400),
        (3.51, 5.0, 401, 500),
    ],
}

# ──────────────────────────────────────────────────────────────
# TRAIN / TEST SPLIT
# ──────────────────────────────────────────────────────────────
TRAIN_TEST_SPLIT = 0.8  # 80% train, 20% test

# ──────────────────────────────────────────────────────────────
# ARIMA CONFIGURATION
# ──────────────────────────────────────────────────────────────
ARIMA_MAX_P = 5
ARIMA_MAX_D = 2
ARIMA_MAX_Q = 5
ARIMA_SEASONAL = True
ARIMA_SEASONAL_PERIOD = 7  # Weekly seasonality (more tractable than 365)

# ──────────────────────────────────────────────────────────────
# LSTM CONFIGURATION
# ──────────────────────────────────────────────────────────────
LSTM_WINDOW_SIZE = 30         # Lookback window in days
LSTM_UNITS_OPTIONS = [32, 64, 128]
LSTM_DROPOUT_OPTIONS = [0.1, 0.2, 0.3]
LSTM_LEARNING_RATE_OPTIONS = [0.001, 0.0005, 0.0001]
LSTM_BATCH_SIZE_OPTIONS = [16, 32, 64]
LSTM_EPOCHS = 100
LSTM_PATIENCE = 15            # EarlyStopping patience

# Default LSTM hyperparameters (used when --skip-tuning)
LSTM_DEFAULT_UNITS_1 = 64
LSTM_DEFAULT_UNITS_2 = 32
LSTM_DEFAULT_DROPOUT = 0.2
LSTM_DEFAULT_LR = 0.001
LSTM_DEFAULT_BATCH_SIZE = 32

# ──────────────────────────────────────────────────────────────
# FEATURE SELECTION
# ──────────────────────────────────────────────────────────────
CORRELATION_THRESHOLD = 0.1   # Min |r| to keep a feature
RFE_N_FEATURES = 8            # Number of features for RFE to select
FEATURE_CONSENSUS_MIN = 2     # Feature must be selected by ≥ N methods

# ──────────────────────────────────────────────────────────────
# VISUALIZATION
# ──────────────────────────────────────────────────────────────
PLOT_STYLE = "seaborn-v0_8-darkgrid"
FIGURE_DPI = 150
COLOR_PALETTE = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7", "#DDA0DD"]

# ──────────────────────────────────────────────────────────────
# RANDOM SEED
# ──────────────────────────────────────────────────────────────
RANDOM_SEED = 42


"""
utils.py — Shared Utilities for AQI Forecasting Pipeline
=========================================================
Contains: AQI calculation (CPCB India), evaluation metrics,
plotting helpers, and LSTM sequence preparation.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os



# ──────────────────────────────────────────────────────────────
# AQI CALCULATION (CPCB India Formula)
# ──────────────────────────────────────────────────────────────

def calculate_sub_index(pollutant: str, concentration: float) -> float:
    """
    Calculate AQI sub-index for a single pollutant using CPCB breakpoints.
    
    Formula: Ip = ((IHi - ILo) / (BPHi - BPLo)) * (Cp - BPLo) + ILo
    
    Parameters
    ----------
    pollutant : str
        Name of the pollutant (must be in AQI_BREAKPOINTS).
    concentration : float
        Measured concentration value.
    
    Returns
    -------
    float
        Sub-index value, or NaN if concentration is missing/invalid.
    """
    if pd.isna(concentration) or concentration < 0:
        return np.nan
    
    if pollutant not in AQI_BREAKPOINTS:
        return np.nan
    
    breakpoints = AQI_BREAKPOINTS[pollutant]
    
    for bp_lo, bp_hi, i_lo, i_hi in breakpoints:
        if bp_lo <= concentration <= bp_hi:
            sub_index = ((i_hi - i_lo) / (bp_hi - bp_lo)) * (concentration - bp_lo) + i_lo
            return round(sub_index, 1)
    
    # If concentration exceeds the highest breakpoint, cap at 500
    if concentration > breakpoints[-1][1]:
        return 500.0
    
    return np.nan


def calculate_aqi(row: pd.Series, pollutant_cols: list = None) -> float:
    """
    Calculate overall AQI from a row of pollutant concentrations.
    
    Rules (CPCB India):
    - AQI = max of all sub-indices
    - Minimum 3 pollutants must have valid data
    - At least one of PM2.5 or PM10 must be present
    
    Parameters
    ----------
    row : pd.Series
        Row containing pollutant concentration values.
    pollutant_cols : list, optional
        List of pollutant column names. Defaults to all AQI_BREAKPOINTS keys.
    
    Returns
    -------
    float
        Calculated AQI value, or NaN if insufficient data.
    """
    if pollutant_cols is None:
        pollutant_cols = list(AQI_BREAKPOINTS.keys())
    
    sub_indices = {}
    for pollutant in pollutant_cols:
        if pollutant in row.index and not pd.isna(row[pollutant]):
            si = calculate_sub_index(pollutant, row[pollutant])
            if not pd.isna(si):
                sub_indices[pollutant] = si
    
    # Check minimum data requirements
    if len(sub_indices) < 3:
        return np.nan
    
    # At least one of PM2.5 or PM10 must be present
    if "PM2.5" not in sub_indices and "PM10" not in sub_indices:
        return np.nan
    
    return max(sub_indices.values())


def get_aqi_category(aqi_value: float) -> str:
    """Return AQI category string based on value."""
    if pd.isna(aqi_value):
        return "Unknown"
    if aqi_value <= 50:
        return "Good"
    elif aqi_value <= 100:
        return "Satisfactory"
    elif aqi_value <= 200:
        return "Moderate"
    elif aqi_value <= 300:
        return "Poor"
    elif aqi_value <= 400:
        return "Very Poor"
    else:
        return "Severe"


AQI_CATEGORY_COLORS = {
    "Good": "#009966",
    "Satisfactory": "#58BC50",
    "Moderate": "#FFDD44",
    "Poor": "#FF8C00",
    "Very Poor": "#CC0033",
    "Severe": "#800020",
    "Unknown": "#999999",
}


# ──────────────────────────────────────────────────────────────
# EVALUATION METRICS
# ──────────────────────────────────────────────────────────────

def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray, model_name: str = "Model") -> dict:
    """
    Compute regression evaluation metrics.
    
    Returns dict with MAE, RMSE, R², MAPE and prints a summary.
    """
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    
    # Remove NaN pairs
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # MAPE (avoid division by zero)
    nonzero_mask = y_true != 0
    if nonzero_mask.sum() > 0:
        mape = np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])) * 100
    else:
        mape = np.nan
    
    metrics = {
        "Model": model_name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R²": round(r2, 4),
        "MAPE (%)": round(mape, 2),
    }
    
    print(f"\n{'='*50}")
    print(f"  {model_name} — Evaluation Metrics")
    print(f"{'='*50}")
    for k, v in metrics.items():
        if k != "Model":
            print(f"  {k:<12}: {v}")
    print(f"{'='*50}\n")
    
    return metrics


# ──────────────────────────────────────────────────────────────
# LSTM SEQUENCE PREPARATION
# ──────────────────────────────────────────────────────────────

def create_sequences(data: np.ndarray, target: np.ndarray, window_size: int):
    """
    Create sliding window sequences for LSTM input.
    
    Parameters
    ----------
    data : np.ndarray
        Feature array of shape (n_samples, n_features).
    target : np.ndarray
        Target array of shape (n_samples,).
    window_size : int
        Number of past time steps to use as input.
    
    Returns
    -------
    X : np.ndarray of shape (n_sequences, window_size, n_features)
    y : np.ndarray of shape (n_sequences,)
    """
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i - window_size:i])
        y.append(target[i])
    return np.array(X), np.array(y)


# ──────────────────────────────────────────────────────────────
# PLOTTING HELPERS
# ──────────────────────────────────────────────────────────────

def setup_plot_style():
    """Set up consistent matplotlib styling for all plots."""
    plt.rcParams.update({
        "figure.dpi": FIGURE_DPI,
        "figure.facecolor": "#0D1117",
        "axes.facecolor": "#161B22",
        "axes.edgecolor": "#30363D",
        "axes.labelcolor": "#C9D1D9",
        "axes.grid": True,
        "grid.color": "#21262D",
        "grid.alpha": 0.6,
        "text.color": "#C9D1D9",
        "xtick.color": "#8B949E",
        "ytick.color": "#8B949E",
        "font.family": "sans-serif",
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "legend.facecolor": "#161B22",
        "legend.edgecolor": "#30363D",
        "legend.fontsize": 9,
        "savefig.bbox": "tight",
        "savefig.facecolor": "#0D1117",
        "savefig.pad_inches": 0.3,
    })


def plot_forecast(y_true, y_pred, dates=None, title="Forecast vs Actual",
                  save_path=None, model_name="Model"):
    """
    Plot actual vs predicted values with residual distribution.
    
    Creates a 2-panel figure: time series comparison + residual histogram.
    """
    setup_plot_style()
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), height_ratios=[3, 1],
                              gridspec_kw={"hspace": 0.3})
    
    # Panel 1: Actual vs Predicted
    ax1 = axes[0]
    x_axis = dates if dates is not None else range(len(y_true))
    ax1.plot(x_axis, y_true, color="#4ECDC4", linewidth=1.2, alpha=0.9, label="Actual AQI")
    ax1.plot(x_axis, y_pred, color="#FF6B6B", linewidth=1.2, alpha=0.8,
             linestyle="--", label=f"{model_name} Predicted")
    ax1.fill_between(x_axis, y_true, y_pred, alpha=0.15, color="#FFEAA7")
    ax1.set_title(title, fontsize=15, fontweight="bold", pad=12)
    ax1.set_ylabel("AQI Value")
    ax1.legend(loc="upper right")
    
    # Panel 2: Residuals
    ax2 = axes[1]
    residuals = np.array(y_true) - np.array(y_pred)
    ax2.bar(x_axis, residuals, color="#45B7D1", alpha=0.6, width=1)
    ax2.axhline(0, color="#FF6B6B", linewidth=0.8, linestyle="--")
    ax2.set_ylabel("Residual")
    ax2.set_xlabel("Date" if dates is not None else "Time Step")
    ax2.set_title("Prediction Residuals", fontsize=12, pad=8)
    
    if save_path:
        plt.savefig(save_path)
        print(f"  📊 Plot saved: {save_path}")
    plt.close()
    
    return fig


def plot_comparison(y_true, predictions_dict, dates=None,
                    title="Model Comparison", save_path=None):
    """
    Plot actual values against multiple model predictions.
    
    Parameters
    ----------
    predictions_dict : dict
        {model_name: y_pred_array}
    """
    setup_plot_style()
    fig, ax = plt.subplots(figsize=(14, 6))
    
    x_axis = dates if dates is not None else range(len(y_true))
    ax.plot(x_axis, y_true, color="#FFFFFF", linewidth=1.8, alpha=0.9,
            label="Actual AQI", zorder=5)
    
    colors = ["#FF6B6B", "#4ECDC4", "#FFEAA7", "#DDA0DD", "#45B7D1"]
    for idx, (name, y_pred) in enumerate(predictions_dict.items()):
        ax.plot(x_axis, y_pred, color=colors[idx % len(colors)],
                linewidth=1.2, alpha=0.8, linestyle="--", label=name)
    
    ax.set_title(title, fontsize=15, fontweight="bold", pad=12)
    ax.set_ylabel("AQI Value")
    ax.set_xlabel("Date" if dates is not None else "Time Step")
    ax.legend(loc="upper right")
    
    if save_path:
        plt.savefig(save_path)
        print(f"  📊 Plot saved: {save_path}")
    plt.close()
    
    return fig


## 2. Synthetic Data Generation

In [ ]:
"""
01_data_generation.py — Synthetic Refinery Pollutant Data Generator
====================================================================
Generates realistic daily pollutant data for 3 Indian Oil Corporation
refinery zones (Mathura, Panipat, Gujarat) with:
  - Seasonal patterns (higher pollution in winter)
  - Weekly cycles (lower on weekends)
  - Long-term trend (gradual improvement)
  - Correlated pollutant spikes
  - ~15% random missingness (MCAR + MAR)
"""

import numpy as np
import pandas as pd
import os

    REFINERY_ZONES, DATE_RANGE_START, DATE_RANGE_END,
    MISSING_FRACTION, DATA_DIR, RANDOM_SEED, ALL_FEATURES
)


def generate_seasonal_component(dates: pd.DatetimeIndex) -> np.ndarray:
    """
    Generate seasonal multiplier: higher in winter (Oct-Feb), lower in monsoon (Jul-Sep).
    Returns array of multipliers centered around 1.0.
    """
    day_of_year = dates.dayofyear.values.astype(float)
    # Peak around day 15 (Jan 15) — winter peak
    seasonal = 1.0 + 0.35 * np.cos(2 * np.pi * (day_of_year - 15) / 365)
    return np.array(seasonal)


def generate_weekly_component(dates: pd.DatetimeIndex) -> np.ndarray:
    """
    Generate weekly cycle: lower on weekends (reduced industrial activity).
    Returns array of multipliers centered around 1.0.
    """
    day_of_week = dates.dayofweek  # 0=Mon, 6=Sun
    weekly = np.where(day_of_week >= 5, 0.85, 1.03)  # Weekends lower
    return weekly


def generate_trend_component(dates: pd.DatetimeIndex) -> np.ndarray:
    """
    Generate gradual downward trend (emission controls improving over years).
    Returns array of multipliers starting at ~1.05 and ending at ~0.92.
    """
    total_days = (dates[-1] - dates[0]).days
    days_elapsed = (dates - dates[0]).days.values.astype(float)
    trend = 1.05 - 0.13 * (days_elapsed / total_days)
    return np.array(trend)


def inject_correlated_spikes(df: pd.DataFrame, zone: str, rng: np.random.Generator) -> pd.DataFrame:
    """
    Inject correlated pollution spikes — when one pollutant spikes,
    related ones spike too (e.g., SO2 ↑ → PM2.5 ↑, PM10 ↑).
    """
    n = len(df)
    n_spikes = int(n * 0.03)  # ~3% of days have spikes
    spike_days = rng.choice(n, size=n_spikes, replace=False)
    
    spike_groups = [
        (["PM2.5", "PM10"], 1.8),          # Particulate spike
        (["SO2", "PM2.5", "NO2"], 1.6),     # Refinery emission spike
        (["CO", "NO2"], 1.5),               # Combustion spike
        (["O3"], 1.7),                       # Photochemical spike (summer)
    ]
    
    for day_idx in spike_days:
        group = spike_groups[rng.integers(0, len(spike_groups))]
        pollutants, multiplier = group
        for p in pollutants:
            if p in df.columns:
                spike_val = multiplier + rng.normal(0, 0.15)
                df.iloc[day_idx, df.columns.get_loc(p)] *= max(spike_val, 1.0)
    
    return df


def inject_missingness(df: pd.DataFrame, fraction: float, rng: np.random.Generator) -> pd.DataFrame:
    """
    Inject ~fraction of missing values using a mix of:
    - MCAR (Missing Completely At Random): ~60% of missing
    - MAR (Missing At Random): ~40% — more missing on high-wind days
    """
    feature_cols = [c for c in df.columns if c not in ["Date", "Zone"]]
    n_total_cells = len(df) * len(feature_cols)
    
    # MCAR: Random missing values
    n_mcar = int(n_total_cells * fraction * 0.6)
    for _ in range(n_mcar):
        row = rng.integers(0, len(df))
        col = feature_cols[rng.integers(0, len(feature_cols))]
        df.at[df.index[row], col] = np.nan
    
    # MAR: More missing on high-wind days (instrument failure)
    if "Wind_Speed" in df.columns:
        high_wind_mask = df["Wind_Speed"] > df["Wind_Speed"].quantile(0.8)
        high_wind_indices = df.index[high_wind_mask]
        n_mar = int(n_total_cells * fraction * 0.4)
        pollutant_cols = [c for c in feature_cols if c not in ["Wind_Speed", "Temperature", "Humidity"]]
        for _ in range(min(n_mar, len(high_wind_indices) * len(pollutant_cols))):
            row = rng.choice(high_wind_indices)
            col = pollutant_cols[rng.integers(0, len(pollutant_cols))]
            df.at[row, col] = np.nan
    
    return df


def generate_zone_data(zone_name: str, zone_params: dict,
                       dates: pd.DatetimeIndex, rng: np.random.Generator) -> pd.DataFrame:
    """Generate synthetic pollutant data for a single refinery zone."""
    n = len(dates)
    
    # Temporal components
    seasonal = generate_seasonal_component(dates)
    weekly = generate_weekly_component(dates)
    trend = generate_trend_component(dates)
    
    data = {"Date": dates, "Zone": zone_name}
    
    for feature, (mean, std) in zone_params.items():
        # Base signal: random noise around mean
        base = rng.normal(mean, std, size=n)
        
        if feature in ["PM2.5", "PM10", "SO2", "NO2", "CO", "NH3", "Pb"]:
            # Apply seasonal, weekly, trend modulation
            signal = np.array(base * seasonal * weekly * trend, dtype=float)
        elif feature == "O3":
            # O3 is inversely seasonal (higher in summer)
            inverse_seasonal = 2.0 - seasonal
            signal = np.array(base * inverse_seasonal * weekly * trend, dtype=float)
        elif feature == "Temperature":
            # Temperature has its own seasonal pattern
            doy = dates.dayofyear.values.astype(float)
            temp_seasonal = mean + (std * 1.2) * np.sin(2 * np.pi * (doy - 100) / 365)
            signal = np.array(temp_seasonal + rng.normal(0, std * 0.3, size=n), dtype=float)
        elif feature == "Humidity":
            # Higher humidity in monsoon (Jul-Sep)
            monsoon = np.where((dates.month >= 7) & (dates.month <= 9), 1.3, 0.9)
            signal = np.array(base * monsoon, dtype=float)
        elif feature == "Wind_Speed":
            signal = np.abs(base).astype(float)  # Wind speed is always positive
        else:
            signal = np.array(base, dtype=float)
        
        # Ensure non-negative values for pollutants
        if feature not in ["Temperature"]:
            signal = np.maximum(signal, 0.01)
        
        # Add autocorrelation (today depends on yesterday)
        for i in range(1, n):
            signal[i] = 0.6 * signal[i] + 0.4 * signal[i - 1]
        
        data[feature] = np.round(signal, 3)
    
    df = pd.DataFrame(data)
    
    # Inject correlated spikes
    df = inject_correlated_spikes(df, zone_name, rng)
    
    return df


def generate_all_data() -> pd.DataFrame:
    """
    Generate synthetic pollutant data for all refinery zones.
    
    Returns
    -------
    pd.DataFrame
        Combined dataframe with columns: Date, Zone, PM2.5, PM10, ..., Wind_Speed
        Includes ~15% missing values.
    """
    print("\n" + "=" * 60)
    print("  Phase 1: SYNTHETIC DATA GENERATION")
    print("=" * 60)
    
    rng = np.random.default_rng(RANDOM_SEED)
    dates = pd.date_range(start=DATE_RANGE_START, end=DATE_RANGE_END, freq="D")
    
    all_zones = []
    for zone_name, zone_params in REFINERY_ZONES.items():
        print(f"\n  🏭 Generating data for {zone_name}...")
        df = generate_zone_data(zone_name, zone_params, dates, rng)
        all_zones.append(df)
        print(f"     ✓ {len(df)} daily records generated")
    
    # Combine all zones
    combined = pd.concat(all_zones, ignore_index=True)
    print(f"\n  📊 Combined dataset: {combined.shape[0]} rows × {combined.shape[1]} columns")
    
    # Save BEFORE injecting missingness (for reference)
    complete_path = os.path.join(DATA_DIR, "complete_data_reference.csv")
    combined.to_csv(complete_path, index=False)
    print(f"  💾 Complete data saved: {complete_path}")
    
    # Inject missingness
    print(f"\n  🔧 Injecting ~{MISSING_FRACTION*100:.0f}% missing values...")
    combined = inject_missingness(combined, MISSING_FRACTION, rng)
    
    # Report missingness
    feature_cols = [c for c in combined.columns if c not in ["Date", "Zone"]]
    missing_pct = combined[feature_cols].isna().mean() * 100
    print(f"\n  Missing value percentages:")
    for col, pct in missing_pct.items():
        bar = "█" * int(pct / 2) + "░" * (25 - int(pct / 2))
        print(f"     {col:<14} {bar} {pct:.1f}%")
    
    # Save raw data with missingness
    raw_path = os.path.join(DATA_DIR, "raw_pollutant_data.csv")
    combined.to_csv(raw_path, index=False)
    print(f"\n  💾 Raw data (with missing values) saved: {raw_path}")
    print(f"\n  ✅ Phase 1 complete: {combined.shape[0]} records across {len(REFINERY_ZONES)} zones")
    
    return combined


if __name__ == "__main__":
    df = generate_all_data()
    print(f"\nSample data:\n{df.head(10)}")


In [ ]:
raw_df = generate_all_data()
display(raw_df.head())

## 3. Data Preprocessing & AQI Calculation

In [ ]:
"""
02_data_preprocessing.py — Data Preprocessing & AQI Calculation
================================================================
Handles:
  - Missing value analysis & visualization
  - Multi-strategy imputation (interpolation, KNN, forward fill)
  - Outlier detection & winsorization
  - AQI computation using CPCB India formula
  - Feature engineering (rolling averages, lags, rate-of-change)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
import os

    DATA_DIR, EDA_PLOTS_DIR, AQI_POLLUTANTS, METEO_FEATURES,
    ALL_FEATURES, RANDOM_SEED
)


def analyze_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Analyze and visualize missing value patterns."""
    feature_cols = [c for c in df.columns if c in ALL_FEATURES]
    
    missing_summary = pd.DataFrame({
        "Column": feature_cols,
        "Missing_Count": [df[c].isna().sum() for c in feature_cols],
        "Missing_Pct": [df[c].isna().mean() * 100 for c in feature_cols],
        "Total_Count": [len(df) for _ in feature_cols],
    })
    missing_summary = missing_summary.sort_values("Missing_Pct", ascending=False)
    
    print("\n  📋 Missing Value Summary:")
    print(missing_summary.to_string(index=False))
    
    # Missing value heatmap
    setup_plot_style()
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Sample for visualization (full data too dense)
    sample_df = df[feature_cols].iloc[::3]  # Every 3rd row
    sns.heatmap(sample_df.isna().T, cbar=False, cmap=["#161B22", "#FF6B6B"],
                ax=ax, yticklabels=True)
    ax.set_title("Missing Data Pattern (white = present, red = missing)",
                 fontsize=13, fontweight="bold", pad=10)
    ax.set_xlabel("Observation Index (sampled)")
    
    path = os.path.join(EDA_PLOTS_DIR, "missing_data_heatmap.png")
    plt.savefig(path)
    plt.close()
    print(f"  📊 Missing data heatmap saved: {path}")
    
    return missing_summary


def impute_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Multi-strategy imputation:
    1. Time-series interpolation for small gaps (≤ 3 consecutive NaNs)
    2. KNN imputation using correlated features for larger gaps
    3. Forward/backward fill as final fallback
    """
    feature_cols = [c for c in df.columns if c in ALL_FEATURES]
    df = df.copy()
    
    print("\n  🔧 Imputation Strategy:")
    
    # Step 1: Time-series interpolation (handles small gaps)
    print("     Step 1/3: Time-series interpolation (limit=3)...")
    for col in feature_cols:
        df[col] = df[col].interpolate(method="linear", limit=3, limit_direction="both")
    
    remaining_before = df[feature_cols].isna().sum().sum()
    print(f"     → Remaining missing values: {remaining_before}")
    
    # Step 2: KNN imputation for larger gaps
    if remaining_before > 0:
        print("     Step 2/3: KNN imputation (k=5)...")
        knn_imputer = KNNImputer(n_neighbors=5, weights="distance")
        df[feature_cols] = knn_imputer.fit_transform(df[feature_cols])
        
        remaining_after = df[feature_cols].isna().sum().sum()
        print(f"     → Remaining missing values: {remaining_after}")
    
    # Step 3: Forward/backward fill (final fallback)
    remaining = df[feature_cols].isna().sum().sum()
    if remaining > 0:
        print("     Step 3/3: Forward/backward fill (fallback)...")
        df[feature_cols] = df[feature_cols].ffill().bfill()
    
    final_missing = df[feature_cols].isna().sum().sum()
    print(f"\n  ✅ Imputation complete. Remaining missing: {final_missing}")
    
    return df


def detect_and_handle_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Detect outliers using IQR method and apply winsorization.
    Pollutant values are clipped to [Q1 - 1.5*IQR, Q3 + 1.5*IQR].
    """
    df = df.copy()
    pollutant_cols = [c for c in df.columns if c in AQI_POLLUTANTS]
    
    print("\n  🔍 Outlier Detection & Handling (IQR method):")
    total_outliers = 0
    
    for col in pollutant_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        outlier_mask = (df[col] < lower) | (df[col] > upper)
        n_outliers = outlier_mask.sum()
        total_outliers += n_outliers
        
        if n_outliers > 0:
            # Winsorize: clip to bounds
            df[col] = df[col].clip(lower=max(lower, 0), upper=upper)
            print(f"     {col:<14}: {n_outliers:>4} outliers clipped to [{max(lower,0):.1f}, {upper:.1f}]")
    
    print(f"\n  ✅ Total outliers handled: {total_outliers}")
    return df


def compute_aqi(df: pd.DataFrame) -> pd.DataFrame:
    """Compute AQI and category for each row using CPCB India formula."""
    df = df.copy()
    
    print("\n  🧮 Computing AQI using CPCB India formula...")
    df["AQI"] = df.apply(lambda row: calculate_aqi(row, AQI_POLLUTANTS), axis=1)
    df["AQI_Category"] = df["AQI"].apply(get_aqi_category)
    
    valid_aqi = df["AQI"].notna().sum()
    print(f"     Valid AQI values: {valid_aqi}/{len(df)} ({valid_aqi/len(df)*100:.1f}%)")
    
    # AQI category distribution
    cat_dist = df["AQI_Category"].value_counts()
    print(f"\n  📊 AQI Category Distribution:")
    for cat, count in cat_dist.items():
        pct = count / len(df) * 100
        print(f"     {cat:<15}: {count:>5} ({pct:.1f}%)")
    
    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create derived features:
    - Rolling averages (7-day, 30-day) for AQI and key pollutants
    - Lag features (1-day, 7-day)
    - Rate of change (day-over-day)
    - Day-of-week, month, season indicators
    """
    df = df.copy()
    
    print("\n  ⚙️  Engineering features...")
    
    # Ensure Date is datetime
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Rolling averages (per zone)
    for col in ["AQI", "PM2.5", "PM10"]:
        if col in df.columns:
            df[f"{col}_MA7"] = df.groupby("Zone")[col].transform(
                lambda x: x.rolling(7, min_periods=1).mean()
            )
            df[f"{col}_MA30"] = df.groupby("Zone")[col].transform(
                lambda x: x.rolling(30, min_periods=1).mean()
            )
    
    # Lag features
    if "AQI" in df.columns:
        df["AQI_lag1"] = df.groupby("Zone")["AQI"].shift(1)
        df["AQI_lag7"] = df.groupby("Zone")["AQI"].shift(7)
        
        # Rate of change
        df["AQI_roc"] = df.groupby("Zone")["AQI"].pct_change()
    
    # Temporal features
    df["DayOfWeek"] = df["Date"].dt.dayofweek
    df["Month"] = df["Date"].dt.month
    df["Season"] = df["Month"].map({
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Summer", 4: "Summer", 5: "Summer",
        6: "Monsoon", 7: "Monsoon", 8: "Monsoon",
        9: "Post-Monsoon", 10: "Post-Monsoon", 11: "Post-Monsoon",
    })
    df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)
    
    # Fill any NaN in new features
    df = df.bfill().ffill()
    
    n_new = len([c for c in df.columns if c not in ALL_FEATURES + ["Date", "Zone", "AQI", "AQI_Category"]])
    print(f"     ✓ {n_new} new features engineered")
    print(f"     ✓ Final shape: {df.shape[0]} rows × {df.shape[1]} columns")
    
    return df


def preprocess_data(df: pd.DataFrame = None) -> pd.DataFrame:
    """
    Full preprocessing pipeline.
    
    Parameters
    ----------
    df : pd.DataFrame, optional
        Raw data. If None, loads from CSV.
    
    Returns
    -------
    pd.DataFrame
        Fully preprocessed data with AQI and engineered features.
    """
    print("\n" + "=" * 60)
    print("  Phase 2: DATA PREPROCESSING")
    print("=" * 60)
    
    if df is None:
        raw_path = os.path.join(DATA_DIR, "raw_pollutant_data.csv")
        print(f"\n  📂 Loading raw data from: {raw_path}")
        df = pd.read_csv(raw_path)
    
    print(f"  📊 Input shape: {df.shape[0]} rows × {df.shape[1]} columns")
    
    # Step 1: Analyze missing values
    analyze_missing_values(df)
    
    # Step 2: Impute missing values
    df = impute_missing_values(df)
    
    # Step 3: Handle outliers
    df = detect_and_handle_outliers(df)
    
    # Step 4: Compute AQI
    df = compute_aqi(df)
    
    # Step 5: Feature engineering
    df = engineer_features(df)
    
    # Save processed data
    processed_path = os.path.join(DATA_DIR, "processed_data.csv")
    df.to_csv(processed_path, index=False)
    print(f"\n  💾 Processed data saved: {processed_path}")
    print(f"  ✅ Phase 2 complete: {df.shape[0]} rows × {df.shape[1]} columns")
    
    return df


if __name__ == "__main__":
    df = preprocess_data()
    print(f"\nProcessed data sample:\n{df.head()}")
    print(f"\nColumns: {list(df.columns)}")


In [ ]:
processed_df = preprocess_data(raw_df)
display(processed_df.head())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
"""
03_eda_visualization.py — Exploratory Data Analysis & Visualization
====================================================================
Generates comprehensive EDA plots:
  - Time series for AQI and pollutants by zone
  - Correlation heatmap
  - Distribution plots (histograms + KDE)
  - Seasonal box plots (monthly)
  - AQI category distribution
  - Zone-wise comparison
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os

    DATA_DIR, EDA_PLOTS_DIR, AQI_POLLUTANTS, METEO_FEATURES,
    COLOR_PALETTE, FIGURE_DPI
)


def plot_aqi_time_series(df: pd.DataFrame):
    """Plot AQI time series for all refinery zones."""
    setup_plot_style()
    zones = df["Zone"].unique()
    fig, axes = plt.subplots(len(zones), 1, figsize=(16, 4 * len(zones)),
                              sharex=True)
    if len(zones) == 1:
        axes = [axes]
    
    colors = ["#FF6B6B", "#4ECDC4", "#45B7D1"]
    
    for idx, (zone, ax) in enumerate(zip(zones, axes)):
        zone_df = df[df["Zone"] == zone].copy()
        zone_df["Date"] = pd.to_datetime(zone_df["Date"])
        
        ax.plot(zone_df["Date"], zone_df["AQI"], color=colors[idx],
                linewidth=0.8, alpha=0.7)
        
        # Add 30-day moving average
        if "AQI_MA30" in zone_df.columns:
            ax.plot(zone_df["Date"], zone_df["AQI_MA30"],
                    color="#FFFFFF", linewidth=1.5, alpha=0.9,
                    label="30-day MA")
        
        # Color bands for AQI categories
        ax.axhspan(0, 50, alpha=0.05, color="#009966")
        ax.axhspan(50, 100, alpha=0.05, color="#58BC50")
        ax.axhspan(100, 200, alpha=0.05, color="#FFDD44")
        ax.axhspan(200, 300, alpha=0.05, color="#FF8C00")
        ax.axhspan(300, 500, alpha=0.05, color="#CC0033")
        
        ax.set_ylabel("AQI")
        ax.set_title(f"🏭 {zone.replace('_', ' ')}", fontsize=13, fontweight="bold")
        ax.legend(loc="upper right")
        ax.set_ylim(0, df["AQI"].quantile(0.99) * 1.1)
    
    axes[-1].set_xlabel("Date")
    fig.suptitle("Air Quality Index — Indian Oil Refinery Zones",
                 fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    
    path = os.path.join(EDA_PLOTS_DIR, "aqi_time_series.png")
    plt.savefig(path)
    plt.close()
    print(f"  📊 AQI time series: {path}")


def plot_pollutant_distributions(df: pd.DataFrame):
    """Plot distribution of each pollutant across zones."""
    setup_plot_style()
    pollutants = [p for p in AQI_POLLUTANTS if p in df.columns]
    n_cols = 4
    n_rows = (len(pollutants) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = axes.flatten()
    
    zone_colors = {"Mathura_Refinery": "#FF6B6B", "Panipat_Refinery": "#4ECDC4",
                   "Gujarat_Refinery": "#45B7D1"}
    
    for idx, pollutant in enumerate(pollutants):
        ax = axes[idx]
        for zone, color in zone_colors.items():
            zone_data = df[df["Zone"] == zone][pollutant].dropna()
            ax.hist(zone_data, bins=40, alpha=0.5, color=color, density=True,
                    label=zone.split("_")[0])
            zone_data.plot.kde(ax=ax, color=color, linewidth=1.5)
        
        ax.set_title(pollutant, fontsize=12, fontweight="bold")
        ax.set_xlabel("Concentration")
        ax.set_ylabel("Density")
        if idx == 0:
            ax.legend(fontsize=8)
    
    # Hide unused axes
    for idx in range(len(pollutants), len(axes)):
        axes[idx].set_visible(False)
    
    fig.suptitle("Pollutant Distributions by Refinery Zone",
                 fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    
    path = os.path.join(EDA_PLOTS_DIR, "pollutant_distributions.png")
    plt.savefig(path)
    plt.close()
    print(f"  📊 Pollutant distributions: {path}")


def plot_correlation_heatmap(df: pd.DataFrame):
    """Plot correlation matrix heatmap of all features."""
    setup_plot_style()
    feature_cols = [c for c in AQI_POLLUTANTS + METEO_FEATURES + ["AQI"] if c in df.columns]
    
    corr = df[feature_cols].corr()
    
    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    
    cmap = sns.diverging_palette(250, 15, s=85, l=40, n=9, center="dark", as_cmap=True)
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap=cmap,
                vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
                linecolor="#30363D", ax=ax,
                cbar_kws={"shrink": 0.8, "label": "Correlation Coefficient"})
    
    ax.set_title("Feature Correlation Matrix",
                 fontsize=15, fontweight="bold", pad=15)
    
    plt.tight_layout()
    path = os.path.join(EDA_PLOTS_DIR, "correlation_heatmap.png")
    plt.savefig(path)
    plt.close()
    print(f"  📊 Correlation heatmap: {path}")


def plot_monthly_boxplots(df: pd.DataFrame):
    """Plot monthly AQI variation as box plots."""
    setup_plot_style()
    fig, ax = plt.subplots(figsize=(14, 6))
    
    df_plot = df.copy()
    df_plot["Month"] = pd.to_datetime(df_plot["Date"]).dt.month
    month_names = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
                   7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
    df_plot["Month_Name"] = df_plot["Month"].map(month_names)
    
    # Order months
    month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    
    box_palette = {m: COLOR_PALETTE[i % len(COLOR_PALETTE)] for i, m in enumerate(month_order)}
    
    sns.boxplot(data=df_plot, x="Month_Name", y="AQI", order=month_order,
                palette=box_palette, ax=ax, flierprops={"marker": ".", "alpha": 0.3},
                linewidth=0.8)
    
    ax.set_title("Monthly AQI Variation Across All Refinery Zones",
                 fontsize=14, fontweight="bold", pad=12)
    ax.set_xlabel("Month")
    ax.set_ylabel("AQI Value")
    
    # Add category reference lines
    for val, label, color in [(50, "Good", "#009966"), (100, "Satisfactory", "#58BC50"),
                               (200, "Moderate", "#FFDD44"), (300, "Poor", "#FF8C00")]:
        ax.axhline(val, color=color, linewidth=0.8, linestyle="--", alpha=0.5)
        ax.text(11.5, val + 5, label, fontsize=7, color=color, ha="right")
    
    plt.tight_layout()
    path = os.path.join(EDA_PLOTS_DIR, "monthly_aqi_boxplots.png")
    plt.savefig(path)
    plt.close()
    print(f"  📊 Monthly boxplots: {path}")


def plot_aqi_category_pie(df: pd.DataFrame):
    """Plot AQI category distribution as a donut chart."""
    setup_plot_style()
    fig, axes = plt.subplots(1, len(df["Zone"].unique()) + 1,
                              figsize=(5 * (len(df["Zone"].unique()) + 1), 5))
    
    # Overall
    cat_counts = df["AQI_Category"].value_counts()
    colors = [AQI_CATEGORY_COLORS.get(c, "#999") for c in cat_counts.index]
    
    axes[0].pie(cat_counts.values, labels=cat_counts.index, colors=colors,
                autopct="%1.1f%%", startangle=90, pctdistance=0.8,
                textprops={"fontsize": 8, "color": "#C9D1D9"})
    centre_circle = plt.Circle((0, 0), 0.55, fc="#0D1117")
    axes[0].add_artist(centre_circle)
    axes[0].set_title("All Zones", fontsize=12, fontweight="bold")
    
    # Per zone
    for idx, zone in enumerate(sorted(df["Zone"].unique()), 1):
        zone_df = df[df["Zone"] == zone]
        cat_counts = zone_df["AQI_Category"].value_counts()
        colors = [AQI_CATEGORY_COLORS.get(c, "#999") for c in cat_counts.index]
        
        axes[idx].pie(cat_counts.values, labels=cat_counts.index, colors=colors,
                      autopct="%1.1f%%", startangle=90, pctdistance=0.8,
                      textprops={"fontsize": 8, "color": "#C9D1D9"})
        centre_circle = plt.Circle((0, 0), 0.55, fc="#0D1117")
        axes[idx].add_artist(centre_circle)
        axes[idx].set_title(zone.replace("_", " "), fontsize=11, fontweight="bold")
    
    fig.suptitle("AQI Category Distribution",
                 fontsize=15, fontweight="bold", y=1.05)
    plt.tight_layout()
    
    path = os.path.join(EDA_PLOTS_DIR, "aqi_category_distribution.png")
    plt.savefig(path)
    plt.close()
    print(f"  📊 AQI category chart: {path}")


def plot_zone_comparison(df: pd.DataFrame):
    """Compare AQI statistics across refinery zones."""
    setup_plot_style()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    zone_colors = ["#FF6B6B", "#4ECDC4", "#45B7D1"]
    zones = sorted(df["Zone"].unique())
    
    # Bar chart: Mean AQI by zone
    mean_aqi = df.groupby("Zone")["AQI"].mean()
    bars = axes[0].bar(range(len(zones)), [mean_aqi[z] for z in zones],
                       color=zone_colors, alpha=0.85, edgecolor="#30363D")
    axes[0].set_xticks(range(len(zones)))
    axes[0].set_xticklabels([z.replace("_", "\n") for z in zones], fontsize=9)
    axes[0].set_ylabel("Mean AQI")
    axes[0].set_title("Average AQI by Refinery Zone", fontsize=13, fontweight="bold")
    
    for bar, zone in zip(bars, zones):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
                     f"{mean_aqi[zone]:.0f}", ha="center", fontsize=10,
                     fontweight="bold", color="#C9D1D9")
    
    # Violin plot: AQI distribution by zone
    parts = axes[1].violinplot([df[df["Zone"] == z]["AQI"].dropna().values for z in zones],
                                showmeans=True, showmedians=True)
    
    for idx, pc in enumerate(parts["bodies"]):
        pc.set_facecolor(zone_colors[idx])
        pc.set_alpha(0.7)
    
    axes[1].set_xticks(range(1, len(zones) + 1))
    axes[1].set_xticklabels([z.replace("_", "\n") for z in zones], fontsize=9)
    axes[1].set_ylabel("AQI")
    axes[1].set_title("AQI Distribution by Refinery Zone", fontsize=13, fontweight="bold")
    
    plt.tight_layout()
    path = os.path.join(EDA_PLOTS_DIR, "zone_comparison.png")
    plt.savefig(path)
    plt.close()
    print(f"  📊 Zone comparison: {path}")


def run_eda(df: pd.DataFrame = None):
    """Run complete EDA pipeline."""
    print("\n" + "=" * 60)
    print("  Phase 3: EXPLORATORY DATA ANALYSIS")
    print("=" * 60)
    
    if df is None:
        processed_path = os.path.join(DATA_DIR, "processed_data.csv")
        print(f"\n  📂 Loading processed data from: {processed_path}")
        df = pd.read_csv(processed_path)
    
    df["Date"] = pd.to_datetime(df["Date"])
    
    print(f"\n  📊 Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"  📊 Zones: {df['Zone'].unique().tolist()}")
    print(f"  📊 Date range: {df['Date'].min()} to {df['Date'].max()}")
    
    # Summary statistics
    print(f"\n  📋 AQI Statistics:")
    aqi_stats = df.groupby("Zone")["AQI"].describe().round(1)
    print(aqi_stats.to_string())
    
    # Generate plots
    print(f"\n  🎨 Generating visualizations...")
    plot_aqi_time_series(df)
    plot_pollutant_distributions(df)
    plot_correlation_heatmap(df)
    plot_monthly_boxplots(df)
    plot_aqi_category_pie(df)
    plot_zone_comparison(df)
    
    print(f"\n  ✅ Phase 3 complete: 6 visualization sets generated in {EDA_PLOTS_DIR}")
    return df


if __name__ == "__main__":
    run_eda()


In [ ]:
run_eda(processed_df)

## 5. Feature Selection

In [ ]:
"""
04_feature_selection.py — Feature Selection for AQI Prediction
===============================================================
Three-stage feature selection:
  1. Correlation analysis (Pearson & Spearman)
  2. Mutual Information scoring
  3. Recursive Feature Elimination (RFE) with Random Forest
  4. Consensus ranking — features selected by ≥ 2 methods
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import (
    mutual_info_regression, RFE, SelectKBest, f_regression
)
from sklearn.preprocessing import StandardScaler
import os

    DATA_DIR, RESULTS_DIR, EDA_PLOTS_DIR, AQI_POLLUTANTS, METEO_FEATURES,
    CORRELATION_THRESHOLD, RFE_N_FEATURES, FEATURE_CONSENSUS_MIN,
    RANDOM_SEED
)


def get_candidate_features(df: pd.DataFrame) -> list:
    """Get list of candidate feature columns for selection."""
    exclude = ["Date", "Zone", "AQI", "AQI_Category", "Season"]
    candidates = [c for c in df.columns if c not in exclude and df[c].dtype in ["float64", "int64", "int32", "float32"]]
    return candidates


def correlation_selection(df: pd.DataFrame, target: str = "AQI",
                          threshold: float = CORRELATION_THRESHOLD) -> dict:
    """
    Stage 1: Correlation-based feature selection.
    Computes Pearson and Spearman correlation with AQI.
    """
    candidates = get_candidate_features(df)
    
    pearson_corr = df[candidates].corrwith(df[target], method="pearson").abs()
    spearman_corr = df[candidates].corrwith(df[target], method="spearman").abs()
    
    # Average of both
    avg_corr = (pearson_corr + spearman_corr) / 2
    
    selected = avg_corr[avg_corr >= threshold].index.tolist()
    
    results = pd.DataFrame({
        "Feature": candidates,
        "Pearson_r": [pearson_corr.get(c, 0) for c in candidates],
        "Spearman_r": [spearman_corr.get(c, 0) for c in candidates],
        "Avg_Correlation": [avg_corr.get(c, 0) for c in candidates],
        "Selected_Corr": [c in selected for c in candidates],
    }).sort_values("Avg_Correlation", ascending=False)
    
    print(f"\n  📊 Correlation Analysis (threshold = {threshold}):")
    print(f"     Selected {len(selected)}/{len(candidates)} features")
    
    return {"results": results, "selected": selected}


def mutual_info_selection(df: pd.DataFrame, target: str = "AQI",
                          top_k: int = RFE_N_FEATURES) -> dict:
    """
    Stage 2: Mutual Information-based feature selection.
    Captures non-linear dependencies.
    """
    candidates = get_candidate_features(df)
    
    X = df[candidates].fillna(0).values
    y = df[target].fillna(df[target].median()).values
    
    mi_scores = mutual_info_regression(X, y, random_state=RANDOM_SEED, n_neighbors=5)
    
    mi_df = pd.DataFrame({
        "Feature": candidates,
        "MI_Score": mi_scores,
    }).sort_values("MI_Score", ascending=False)
    
    selected = mi_df.head(top_k)["Feature"].tolist()
    mi_df["Selected_MI"] = mi_df["Feature"].isin(selected)
    
    print(f"\n  📊 Mutual Information Analysis (top {top_k}):")
    print(f"     Selected: {selected}")
    
    return {"results": mi_df, "selected": selected}


def rfe_selection(df: pd.DataFrame, target: str = "AQI",
                  n_features: int = RFE_N_FEATURES) -> dict:
    """
    Stage 3: Recursive Feature Elimination with Random Forest.
    """
    candidates = get_candidate_features(df)
    
    X = df[candidates].fillna(0)
    y = df[target].fillna(df[target].median())
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Random Forest estimator
    rf = RandomForestRegressor(
        n_estimators=100, max_depth=10,
        random_state=RANDOM_SEED, n_jobs=-1
    )
    
    rfe = RFE(estimator=rf, n_features_to_select=n_features, step=1)
    rfe.fit(X_scaled, y)
    
    rfe_df = pd.DataFrame({
        "Feature": candidates,
        "RFE_Rank": rfe.ranking_,
        "Selected_RFE": rfe.support_,
    }).sort_values("RFE_Rank")
    
    selected = rfe_df[rfe_df["Selected_RFE"]]["Feature"].tolist()
    
    # Also get feature importances from the fitted RF
    rf.fit(X_scaled, y)
    importances = rf.feature_importances_
    rfe_df["RF_Importance"] = importances
    
    print(f"\n  📊 RFE Analysis (Random Forest, select {n_features}):")
    print(f"     Selected: {selected}")
    
    return {"results": rfe_df, "selected": selected}


def consensus_selection(corr_selected: list, mi_selected: list,
                        rfe_selected: list, min_votes: int = FEATURE_CONSENSUS_MIN) -> list:
    """
    Consensus ranking: Keep features selected by ≥ min_votes methods.
    """
    all_features = set(corr_selected + mi_selected + rfe_selected)
    
    consensus = {}
    for feat in all_features:
        votes = sum([
            feat in corr_selected,
            feat in mi_selected,
            feat in rfe_selected,
        ])
        consensus[feat] = votes
    
    selected = [f for f, v in consensus.items() if v >= min_votes]
    
    print(f"\n  📊 Consensus Selection (≥ {min_votes} votes):")
    print(f"     Selected {len(selected)} features:")
    for feat in sorted(selected):
        votes = consensus[feat]
        methods = []
        if feat in corr_selected:
            methods.append("Corr")
        if feat in mi_selected:
            methods.append("MI")
        if feat in rfe_selected:
            methods.append("RFE")
        print(f"       {feat:<20} — {votes}/3 votes ({', '.join(methods)})")
    
    return selected


def plot_feature_importance(corr_results: pd.DataFrame, mi_results: pd.DataFrame,
                            rfe_results: pd.DataFrame, final_selected: list):
    """Plot comprehensive feature importance visualization."""
    setup_plot_style()
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Correlation scores
    ax = axes[0, 0]
    data = corr_results.sort_values("Avg_Correlation", ascending=True).tail(15)
    colors = ["#4ECDC4" if f in final_selected else "#30363D" for f in data["Feature"]]
    ax.barh(data["Feature"], data["Avg_Correlation"], color=colors, edgecolor="#21262D")
    ax.set_title("Avg Correlation with AQI", fontsize=12, fontweight="bold")
    ax.set_xlabel("Correlation (|r|)")
    
    # Plot 2: Mutual Information scores
    ax = axes[0, 1]
    data = mi_results.sort_values("MI_Score", ascending=True).tail(15)
    colors = ["#FF6B6B" if f in final_selected else "#30363D" for f in data["Feature"]]
    ax.barh(data["Feature"], data["MI_Score"], color=colors, edgecolor="#21262D")
    ax.set_title("Mutual Information Score", fontsize=12, fontweight="bold")
    ax.set_xlabel("MI Score")
    
    # Plot 3: RF Feature Importance
    ax = axes[1, 0]
    data = rfe_results.sort_values("RF_Importance", ascending=True).tail(15)
    colors = ["#FFEAA7" if f in final_selected else "#30363D" for f in data["Feature"]]
    ax.barh(data["Feature"], data["RF_Importance"], color=colors, edgecolor="#21262D")
    ax.set_title("Random Forest Importance", fontsize=12, fontweight="bold")
    ax.set_xlabel("Importance")
    
    # Plot 4: Consensus summary
    ax = axes[1, 1]
    all_features = list(set(
        corr_results["Feature"].tolist() +
        mi_results["Feature"].tolist()
    ))
    
    consensus_data = []
    for f in all_features:
        votes = 0
        if f in corr_results[corr_results["Selected_Corr"]]["Feature"].values:
            votes += 1
        if f in mi_results[mi_results["Selected_MI"]]["Feature"].values:
            votes += 1
        if f in rfe_results[rfe_results["Selected_RFE"]]["Feature"].values:
            votes += 1
        consensus_data.append({"Feature": f, "Votes": votes})
    
    cons_df = pd.DataFrame(consensus_data).sort_values("Votes", ascending=True).tail(15)
    colors = ["#45B7D1" if f in final_selected else "#30363D" for f in cons_df["Feature"]]
    ax.barh(cons_df["Feature"], cons_df["Votes"], color=colors, edgecolor="#21262D")
    ax.set_title("Consensus Votes (3 methods)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Number of Methods Selecting Feature")
    ax.set_xticks([0, 1, 2, 3])
    
    fig.suptitle("Feature Selection Analysis — Highlighted = Final Selected",
                 fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    
    path = os.path.join(EDA_PLOTS_DIR, "feature_importance.png")
    plt.savefig(path)
    plt.close()
    print(f"\n  📊 Feature importance plot: {path}")


def run_feature_selection(df: pd.DataFrame = None) -> list:
    """
    Run complete feature selection pipeline.
    
    Returns
    -------
    list
        Final selected feature names.
    """
    print("\n" + "=" * 60)
    print("  Phase 4: FEATURE SELECTION")
    print("=" * 60)
    
    if df is None:
        processed_path = os.path.join(DATA_DIR, "processed_data.csv")
        df = pd.read_csv(processed_path)
    
    # Filter to numeric columns with valid AQI
    df = df[df["AQI"].notna()].copy()
    
    print(f"\n  📊 Working with {len(get_candidate_features(df))} candidate features")
    
    # Stage 1: Correlation
    corr_result = correlation_selection(df)
    
    # Stage 2: Mutual Information
    mi_result = mutual_info_selection(df)
    
    # Stage 3: RFE
    rfe_result = rfe_selection(df)
    
    # Consensus
    final_selected = consensus_selection(
        corr_result["selected"],
        mi_result["selected"],
        rfe_result["selected"]
    )
    
    # Visualization
    plot_feature_importance(
        corr_result["results"],
        mi_result["results"],
        rfe_result["results"],
        final_selected
    )
    
    # Save results
    results_path = os.path.join(RESULTS_DIR, "feature_selection.csv")
    
    # Merge all results
    merged = corr_result["results"].merge(
        mi_result["results"], on="Feature", how="outer"
    ).merge(
        rfe_result["results"], on="Feature", how="outer"
    )
    merged["Final_Selected"] = merged["Feature"].isin(final_selected)
    merged.to_csv(results_path, index=False)
    print(f"\n  💾 Feature selection results: {results_path}")
    
    print(f"\n  ✅ Phase 4 complete: {len(final_selected)} features selected")
    print(f"     Final features: {final_selected}")
    
    return final_selected


if __name__ == "__main__":
    selected = run_feature_selection()


In [ ]:
selected_features = run_feature_selection(processed_df)
print("Selected Features:", selected_features)

## 6. Seasonal Decomposition & Stationarity

In [ ]:
"""
05_seasonal_decomposition.py — Seasonal Decomposition & Stationarity
======================================================================
Performs:
  - STL decomposition (Seasonal-Trend via LOESS)
  - Classical additive/multiplicative decomposition
  - Stationarity tests (ADF, KPSS)
  - ACF/PACF plots for ARIMA order guidance
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
import os


warnings.filterwarnings("ignore")


def test_stationarity(series: pd.Series, name: str = "Series") -> dict:
    """
    Run ADF and KPSS stationarity tests.
    
    Returns dict with test statistics and conclusions.
    """
    results = {"name": name}
    
    # ADF Test (H0: unit root exists → non-stationary)
    adf_result = adfuller(series.dropna(), autolag="AIC")
    results["ADF_Statistic"] = round(adf_result[0], 4)
    results["ADF_p_value"] = round(adf_result[1], 6)
    results["ADF_Stationary"] = adf_result[1] < 0.05
    
    # KPSS Test (H0: series is stationary)
    try:
        kpss_result = kpss(series.dropna(), regression="c", nlags="auto")
        results["KPSS_Statistic"] = round(kpss_result[0], 4)
        results["KPSS_p_value"] = round(kpss_result[1], 6)
        results["KPSS_Stationary"] = kpss_result[1] > 0.05
    except Exception:
        results["KPSS_Statistic"] = np.nan
        results["KPSS_p_value"] = np.nan
        results["KPSS_Stationary"] = None
    
    # Overall conclusion
    adf_stat = results["ADF_Stationary"]
    kpss_stat = results.get("KPSS_Stationary")
    
    if adf_stat and kpss_stat:
        results["Conclusion"] = "Stationary"
    elif not adf_stat and not kpss_stat:
        results["Conclusion"] = "Non-Stationary"
    elif adf_stat and not kpss_stat:
        results["Conclusion"] = "Trend-Stationary"
    else:
        results["Conclusion"] = "Difference-Stationary"
    
    return results


def perform_stl_decomposition(series: pd.Series, period: int = ARIMA_SEASONAL_PERIOD,
                               name: str = "AQI"):
    """
    Perform STL decomposition and create visualization.
    
    Returns trend, seasonal, and residual components.
    """
    setup_plot_style()
    
    # STL decomposition
    stl = STL(series, period=period, robust=True)
    result = stl.fit()
    
    # Plot
    fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
    
    components = [
        (series, "Original", "#FFFFFF"),
        (result.trend, "Trend", "#4ECDC4"),
        (result.seasonal, "Seasonal", "#FF6B6B"),
        (result.resid, "Residual", "#FFEAA7"),
    ]
    
    for ax, (data, title, color) in zip(axes, components):
        ax.plot(data.index, data.values, color=color, linewidth=0.8, alpha=0.85)
        ax.set_ylabel(title, fontsize=11, fontweight="bold")
        if title == "Residual":
            ax.axhline(0, color="#FF6B6B", linewidth=0.5, linestyle="--", alpha=0.5)
    
    axes[0].set_title(f"STL Decomposition — {name} (period={period})",
                      fontsize=14, fontweight="bold", pad=12)
    axes[-1].set_xlabel("Date")
    
    plt.tight_layout()
    path = os.path.join(DECOMP_PLOTS_DIR, f"stl_decomposition_{name.lower()}.png")
    plt.savefig(path)
    plt.close()
    print(f"  📊 STL decomposition: {path}")
    
    return result


def plot_acf_pacf(series: pd.Series, name: str = "AQI", lags: int = 50):
    """Plot ACF and PACF for ARIMA order selection."""
    setup_plot_style()
    fig, axes = plt.subplots(2, 1, figsize=(14, 7))
    
    plot_acf(series.dropna(), lags=lags, ax=axes[0],
             color="#4ECDC4", vlines_kwargs={"colors": "#4ECDC4"})
    axes[0].set_title(f"Autocorrelation Function (ACF) — {name}",
                      fontsize=13, fontweight="bold")
    
    plot_pacf(series.dropna(), lags=lags, ax=axes[1],
              color="#FF6B6B", vlines_kwargs={"colors": "#FF6B6B"})
    axes[1].set_title(f"Partial Autocorrelation (PACF) — {name}",
                      fontsize=13, fontweight="bold")
    
    for ax in axes:
        ax.axhline(0, color="#C9D1D9", linewidth=0.5)
    
    plt.tight_layout()
    path = os.path.join(DECOMP_PLOTS_DIR, f"acf_pacf_{name.lower()}.png")
    plt.savefig(path)
    plt.close()
    print(f"  📊 ACF/PACF plots: {path}")


def run_seasonal_decomposition(df: pd.DataFrame = None, zone: str = None) -> dict:
    """
    Run complete seasonal decomposition analysis.
    
    Parameters
    ----------
    df : pd.DataFrame, optional
        Processed data. If None, loads from CSV.
    zone : str, optional
        Specific zone to analyze. If None, uses first zone.
    
    Returns
    -------
    dict with decomposition results and stationarity info.
    """
    print("\n" + "=" * 60)
    print("  Phase 5: SEASONAL DECOMPOSITION")
    print("=" * 60)
    
    if df is None:
        processed_path = os.path.join(DATA_DIR, "processed_data.csv")
        df = pd.read_csv(processed_path)
    
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Select zone
    if zone is None:
        zone = df["Zone"].unique()[0]
    
    print(f"\n  🏭 Analyzing zone: {zone}")
    
    zone_df = df[df["Zone"] == zone].copy()
    zone_df = zone_df.set_index("Date").sort_index()
    aqi_series = zone_df["AQI"].dropna()
    
    print(f"  📊 AQI series length: {len(aqi_series)} observations")
    
    # ─── Stationarity Tests ───
    print("\n  🧪 Stationarity Tests:")
    
    # Original series
    stat_original = test_stationarity(aqi_series, "AQI (Original)")
    print(f"\n     Original AQI:")
    print(f"       ADF: statistic={stat_original['ADF_Statistic']}, "
          f"p={stat_original['ADF_p_value']} → {'Stationary' if stat_original['ADF_Stationary'] else 'Non-Stationary'}")
    print(f"       KPSS: statistic={stat_original['KPSS_Statistic']}, "
          f"p={stat_original['KPSS_p_value']} → {'Stationary' if stat_original['KPSS_Stationary'] else 'Non-Stationary'}")
    print(f"       Conclusion: {stat_original['Conclusion']}")
    
    # First difference
    aqi_diff = aqi_series.diff().dropna()
    stat_diff = test_stationarity(aqi_diff, "AQI (1st Difference)")
    print(f"\n     First Difference:")
    print(f"       ADF: statistic={stat_diff['ADF_Statistic']}, "
          f"p={stat_diff['ADF_p_value']} → {'Stationary' if stat_diff['ADF_Stationary'] else 'Non-Stationary'}")
    print(f"       Conclusion: {stat_diff['Conclusion']}")
    
    # ─── STL Decomposition ───
    print("\n  📈 STL Decomposition:")
    stl_result = perform_stl_decomposition(aqi_series, period=ARIMA_SEASONAL_PERIOD, name="AQI")
    
    # Stationarity of residual component
    stat_resid = test_stationarity(stl_result.resid.dropna(), "STL Residual")
    print(f"\n     STL Residual stationarity: {stat_resid['Conclusion']}")
    
    # ─── ACF / PACF ───
    print("\n  📊 ACF/PACF Analysis:")
    plot_acf_pacf(aqi_series, "AQI_Original")
    plot_acf_pacf(aqi_diff, "AQI_Differenced")
    
    # ─── Summary ───
    d_suggested = 0 if stat_original["ADF_Stationary"] else 1
    print(f"\n  💡 Suggested ARIMA differencing order (d): {d_suggested}")
    print(f"     (Based on ADF test — {'original is stationary' if d_suggested == 0 else 'first difference needed'})")
    
    results = {
        "zone": zone,
        "stationarity_original": stat_original,
        "stationarity_differenced": stat_diff,
        "stationarity_residual": stat_resid,
        "stl_result": stl_result,
        "suggested_d": d_suggested,
    }
    
    print(f"\n  ✅ Phase 5 complete: Decomposition and stationarity analysis done")
    
    return results


if __name__ == "__main__":
    results = run_seasonal_decomposition()


In [ ]:
ZONE_TO_ANALYZE = "Mathura_Refinery"
decomp_results = run_seasonal_decomposition(processed_df, zone=ZONE_TO_ANALYZE)

## 7. ARIMA Modeling

In [ ]:
"""
06_arima_model.py — ARIMA/SARIMA Model for AQI Forecasting
============================================================
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA as StatsARIMA
import pmdarima as pm
from pmdarima import auto_arima
import joblib
import warnings
import os

    DATA_DIR, MODELS_DIR, MODEL_PLOTS_DIR, RESULTS_DIR,
    TRAIN_TEST_SPLIT, ARIMA_MAX_P, ARIMA_MAX_D, ARIMA_MAX_Q,
    ARIMA_SEASONAL, ARIMA_SEASONAL_PERIOD, RANDOM_SEED
)

warnings.filterwarnings("ignore")


def fit_auto_arima(train_series, seasonal=ARIMA_SEASONAL, seasonal_period=ARIMA_SEASONAL_PERIOD):
    print("\n  Running Auto-ARIMA (stepwise search)...")
    model = auto_arima(
        train_series, start_p=0, max_p=ARIMA_MAX_P, start_q=0, max_q=ARIMA_MAX_Q,
        max_d=ARIMA_MAX_D, seasonal=seasonal, m=seasonal_period if seasonal else 1,
        start_P=0, max_P=2, start_Q=0, max_Q=2, max_D=1,
        trace=True, error_action="ignore", suppress_warnings=True,
        stepwise=True, information_criterion="aic", n_jobs=-1,
    )
    print(f"\n  Best ARIMA order: {model.order}")
    if seasonal:
        print(f"  Seasonal order: {model.seasonal_order}")
    print(f"  AIC: {model.aic():.2f}, BIC: {model.bic():.2f}")
    return model


def plot_arima_diagnostics(model, save_dir=MODEL_PLOTS_DIR):
    setup_plot_style()
    residuals = model.resid()
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(residuals, color="#4ECDC4", linewidth=0.7, alpha=0.8)
    axes[0, 0].axhline(0, color="#FF6B6B", linewidth=0.8, linestyle="--")
    axes[0, 0].set_title("Residuals Over Time", fontsize=12, fontweight="bold")

    axes[0, 1].hist(residuals, bins=50, color="#FF6B6B", alpha=0.7, edgecolor="#30363D", density=True)
    from scipy import stats
    x = np.linspace(residuals.min(), residuals.max(), 100)
    axes[0, 1].plot(x, stats.norm.pdf(x, residuals.mean(), residuals.std()), color="#FFEAA7", linewidth=2)
    axes[0, 1].set_title("Residual Distribution", fontsize=12, fontweight="bold")

    stats.probplot(residuals, dist="norm", plot=axes[1, 0])
    axes[1, 0].get_lines()[0].set_color("#4ECDC4")
    axes[1, 0].get_lines()[1].set_color("#FF6B6B")
    axes[1, 0].set_title("Q-Q Plot", fontsize=12, fontweight="bold")

    from statsmodels.graphics.tsaplots import plot_acf
    plot_acf(residuals, lags=40, ax=axes[1, 1], color="#45B7D1", vlines_kwargs={"colors": "#45B7D1"})
    axes[1, 1].set_title("Residual ACF", fontsize=12, fontweight="bold")

    fig.suptitle("ARIMA Diagnostics", fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "arima_diagnostics.png"))
    plt.close()


def train_arima(df=None, zone=None):
    print("\n" + "=" * 60)
    print("  Phase 6: ARIMA MODEL TRAINING")
    print("=" * 60)

    if df is None:
        df = pd.read_csv(os.path.join(DATA_DIR, "processed_data.csv"))
    df["Date"] = pd.to_datetime(df["Date"])

    if zone is None:
        zone = df["Zone"].unique()[0]
    print(f"\n  Training zone: {zone}")

    zone_df = df[df["Zone"] == zone].set_index("Date").sort_index()
    aqi_series = zone_df["AQI"].dropna()

    split_idx = int(len(aqi_series) * TRAIN_TEST_SPLIT)
    train = aqi_series.iloc[:split_idx]
    test = aqi_series.iloc[split_idx:]
    print(f"  Train: {len(train)}, Test: {len(test)}")

    model = fit_auto_arima(train)

    forecasts, conf_int = model.predict(n_periods=len(test), return_conf_int=True)
    forecast_series = pd.Series(forecasts, index=test.index)

    train_fitted = model.predict_in_sample()
    train_residuals = train.values - train_fitted
    test_residuals = test.values - forecasts

    metrics = evaluate_model(test.values, forecasts, "ARIMA")

    plot_forecast(test.values, forecasts, dates=test.index,
                  title=f"ARIMA Forecast — {zone.replace('_', ' ')}",
                  save_path=os.path.join(MODEL_PLOTS_DIR, "arima_forecast.png"), model_name="ARIMA")

    # Forecast with confidence interval plot
    setup_plot_style()
    fig, ax = plt.subplots(figsize=(14, 5))
    context = aqi_series.iloc[max(0, split_idx - 60):split_idx]
    ax.plot(context.index, context.values, color="#8B949E", linewidth=0.8, alpha=0.7, label="Training")
    ax.plot(test.index, test.values, color="#4ECDC4", linewidth=1.0, label="Actual")
    ax.plot(test.index, forecasts, color="#FF6B6B", linewidth=1.2, linestyle="--", label="ARIMA")
    ax.fill_between(test.index, conf_int[:, 0], conf_int[:, 1], alpha=0.15, color="#FF6B6B", label="95% CI")
    ax.axvline(train.index[-1], color="#FFEAA7", linewidth=1, linestyle=":", alpha=0.7)
    ax.set_title(f"ARIMA Forecast — {zone.replace('_', ' ')}", fontsize=14, fontweight="bold")
    ax.set_ylabel("AQI"); ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_PLOTS_DIR, "arima_forecast_ci.png"))
    plt.close()

    plot_arima_diagnostics(model)

    joblib.dump(model, os.path.join(MODELS_DIR, "arima_model.pkl"))

    residuals_df = pd.DataFrame({
        "Date": list(train.index) + list(test.index),
        "Actual_AQI": list(train.values) + list(test.values),
        "ARIMA_Predicted": list(train_fitted) + list(forecasts),
        "Residual": list(train_residuals) + list(test_residuals),
        "Split": ["train"] * len(train) + ["test"] * len(test),
    })
    residuals_df.to_csv(os.path.join(DATA_DIR, "arima_residuals.csv"), index=False)

    print(f"\n  Phase 6 complete: ARIMA{model.order} trained")
    return {
        "model": model, "train": train, "test": test, "forecasts": forecast_series,
        "train_residuals": train_residuals, "test_residuals": test_residuals,
        "metrics": metrics, "zone": zone, "order": model.order,
    }

if __name__ == "__main__":
    train_arima()


In [ ]:
arima_results = train_arima(processed_df, zone=ZONE_TO_ANALYZE)

## 8. LSTM Modeling (on ARIMA Residuals)

In [ ]:
"""
07_lstm_model.py — LSTM Model for ARIMA Residual Prediction
=============================================================
Trains LSTM on ARIMA residuals + multivariate features.
Supports hyperparameter tuning via Keras Tuner.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import joblib
import os

    DATA_DIR, MODELS_DIR, MODEL_PLOTS_DIR, RESULTS_DIR,
    LSTM_WINDOW_SIZE, LSTM_EPOCHS, LSTM_PATIENCE, RANDOM_SEED,
    LSTM_DEFAULT_UNITS_1, LSTM_DEFAULT_UNITS_2, LSTM_DEFAULT_DROPOUT,
    LSTM_DEFAULT_LR, LSTM_DEFAULT_BATCH_SIZE, TRAIN_TEST_SPLIT,
    AQI_POLLUTANTS, METEO_FEATURES
)

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def build_lstm_model(input_shape, units_1=LSTM_DEFAULT_UNITS_1, units_2=LSTM_DEFAULT_UNITS_2,
                     dropout=LSTM_DEFAULT_DROPOUT, lr=LSTM_DEFAULT_LR):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(units_1, return_sequences=True),
        Dropout(dropout),
        LSTM(units_2, return_sequences=False),
        Dropout(dropout),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return model


def prepare_lstm_data(df, selected_features, window_size=LSTM_WINDOW_SIZE):
    """Prepare data for LSTM: scale features, create sequences."""
    residuals_path = os.path.join(DATA_DIR, "arima_residuals.csv")
    residuals_df = pd.read_csv(residuals_path)
    residuals_df["Date"] = pd.to_datetime(residuals_df["Date"])

    df["Date"] = pd.to_datetime(df["Date"])
    zone = df["Zone"].unique()[0]
    zone_df = df[df["Zone"] == zone].copy().set_index("Date").sort_index()

    # Align features with residuals
    available_feats = [f for f in selected_features if f in zone_df.columns and f != "AQI"]
    if not available_feats:
        available_feats = [f for f in AQI_POLLUTANTS + METEO_FEATURES if f in zone_df.columns][:5]

    merged = residuals_df.set_index("Date").join(zone_df[available_feats], how="inner")
    merged = merged.dropna()

    # Separate train/test
    train_mask = merged["Split"] == "train"
    test_mask = merged["Split"] == "test"
    train_data = merged[train_mask]
    test_data = merged[test_mask]

    feature_cols = available_feats + ["Residual"]

    # Scale
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train_data[feature_cols])
    test_scaled = scaler.transform(test_data[feature_cols])

    # Target scaler for residuals
    target_scaler = MinMaxScaler()
    train_target = target_scaler.fit_transform(train_data[["Residual"]])
    test_target = target_scaler.transform(test_data[["Residual"]])

    # Create sequences
    X_train, y_train = create_sequences(train_scaled, train_target.flatten(), window_size)
    X_test, y_test = create_sequences(test_scaled, test_target.flatten(), window_size)

    print(f"  LSTM Input shapes: X_train={X_train.shape}, X_test={X_test.shape}")
    print(f"  Features used: {available_feats}")

    return {
        "X_train": X_train, "y_train": y_train,
        "X_test": X_test, "y_test": y_test,
        "scaler": scaler, "target_scaler": target_scaler,
        "feature_cols": feature_cols, "available_feats": available_feats,
        "test_dates": test_data.index[window_size:],
        "test_actual_residuals": test_data["Residual"].values[window_size:],
    }


def run_hyperparameter_tuning(X_train, y_train, X_val, y_val):
    """Hyperparameter tuning via Keras Tuner (RandomSearch)."""
    try:
        import keras_tuner as kt
    except ImportError:
        print("  keras-tuner not installed, using defaults")
        return None

    print("\n  Running hyperparameter tuning...")

    def build_model(hp):
        model = Sequential([
            Input(shape=(X_train.shape[1], X_train.shape[2])),
            LSTM(hp.Choice("units_1", [32, 64, 128]), return_sequences=True),
            Dropout(hp.Choice("dropout", [0.1, 0.2, 0.3])),
            LSTM(hp.Choice("units_2", [16, 32, 64]), return_sequences=False),
            Dropout(hp.Choice("dropout_2", [0.1, 0.2, 0.3])),
            Dense(16, activation="relu"),
            Dense(1),
        ])
        lr = hp.Choice("lr", [0.001, 0.0005, 0.0001])
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss="mse", metrics=["mae"])
        return model

    tuner = kt.RandomSearch(
        build_model, objective="val_loss", max_trials=10, seed=RANDOM_SEED,
        directory=os.path.join(MODELS_DIR, "tuner"), project_name="lstm_aqi",
        overwrite=True,
    )

    tuner.search(X_train, y_train, epochs=30, batch_size=32,
                 validation_data=(X_val, y_val), verbose=0,
                 callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])

    best_hp = tuner.get_best_hyperparameters(1)[0]
    print(f"  Best HP: units_1={best_hp.get('units_1')}, units_2={best_hp.get('units_2')}, "
          f"dropout={best_hp.get('dropout')}, lr={best_hp.get('lr')}")
    return best_hp


def train_lstm(df=None, selected_features=None, skip_tuning=False):
    print("\n" + "=" * 60)
    print("  Phase 7a: LSTM MODEL TRAINING")
    print("=" * 60)

    if df is None:
        df = pd.read_csv(os.path.join(DATA_DIR, "processed_data.csv"))
    if selected_features is None:
        sel_path = os.path.join(RESULTS_DIR, "feature_selection.csv")
        if os.path.exists(sel_path):
            sel_df = pd.read_csv(sel_path)
            selected_features = sel_df[sel_df["Final_Selected"] == True]["Feature"].tolist()
        else:
            selected_features = AQI_POLLUTANTS + METEO_FEATURES

    data = prepare_lstm_data(df, selected_features)
    X_train, y_train = data["X_train"], data["y_train"]
    X_test, y_test = data["X_test"], data["y_test"]

    # Hyperparameter tuning or defaults
    best_hp = None
    if not skip_tuning:
        best_hp = run_hyperparameter_tuning(X_train, y_train, X_test, y_test)

    if best_hp:
        model = build_lstm_model(
            (X_train.shape[1], X_train.shape[2]),
            units_1=best_hp.get("units_1"), units_2=best_hp.get("units_2"),
            dropout=best_hp.get("dropout"), lr=best_hp.get("lr"),
        )
        batch_size = 32
    else:
        model = build_lstm_model((X_train.shape[1], X_train.shape[2]))
        batch_size = LSTM_DEFAULT_BATCH_SIZE

    print("\n  Model architecture:")
    model.summary(print_fn=lambda x: print(f"     {x}"))

    callbacks = [
        EarlyStopping(patience=LSTM_PATIENCE, restore_best_weights=True, monitor="val_loss"),
        ReduceLROnPlateau(factor=0.5, patience=7, min_lr=1e-6, monitor="val_loss"),
    ]

    print(f"\n  Training LSTM (epochs={LSTM_EPOCHS}, batch={batch_size})...")
    history = model.fit(
        X_train, y_train, epochs=LSTM_EPOCHS, batch_size=batch_size,
        validation_data=(X_test, y_test), callbacks=callbacks, verbose=1,
    )

    # Predict
    y_pred_scaled = model.predict(X_test, verbose=0).flatten()

    # Inverse transform
    target_scaler = data["target_scaler"]
    y_pred_residuals = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_actual_residuals = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

    metrics = evaluate_model(y_actual_residuals, y_pred_residuals, "LSTM (Residuals)")

    # Training history plot
    setup_plot_style()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history["loss"], color="#FF6B6B", label="Train Loss")
    axes[0].plot(history.history["val_loss"], color="#4ECDC4", label="Val Loss")
    axes[0].set_title("Training Loss", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE"); axes[0].legend()

    axes[1].plot(history.history["mae"], color="#FF6B6B", label="Train MAE")
    axes[1].plot(history.history["val_mae"], color="#4ECDC4", label="Val MAE")
    axes[1].set_title("Training MAE", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MAE"); axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_PLOTS_DIR, "lstm_training_history.png"))
    plt.close()

    model.save(os.path.join(MODELS_DIR, "lstm_model.keras"))
    joblib.dump(data["scaler"], os.path.join(MODELS_DIR, "lstm_scaler.pkl"))
    joblib.dump(data["target_scaler"], os.path.join(MODELS_DIR, "lstm_target_scaler.pkl"))

    print(f"\n  Phase 7a complete: LSTM trained ({len(history.history['loss'])} epochs)")
    return {
        "model": model, "history": history, "metrics": metrics,
        "y_pred_residuals": y_pred_residuals, "y_actual_residuals": y_actual_residuals,
        "test_dates": data["test_dates"], "data": data,
    }

if __name__ == "__main__":
    train_lstm(skip_tuning=True)


In [ ]:
# For faster execution in the notebook, we'll set epochs to 15 and skip Keras Tuner
LSTM_EPOCHS = 15
LSTM_PATIENCE = 5
lstm_results = train_lstm(processed_df, selected_features, skip_tuning=True)

## 9. Hybrid Combination & Future Forecast

In [ ]:
"""
08_hybrid_forecast.py — Hybrid ARIMA-LSTM Forecast & Evaluation
================================================================
Combines ARIMA + LSTM predictions and compares against standalone models.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import os

    DATA_DIR, MODELS_DIR, MODEL_PLOTS_DIR, RESULTS_DIR, TRAIN_TEST_SPLIT
)


def combine_hybrid_forecast(arima_results, lstm_results):
    """Combine ARIMA predictions + LSTM residual predictions."""
    print("\n" + "=" * 60)
    print("  Phase 7b: HYBRID ARIMA-LSTM COMBINATION")
    print("=" * 60)

    # ARIMA test predictions
    arima_test = arima_results["test"]
    arima_forecasts = arima_results["forecasts"]

    # LSTM residual predictions (shorter due to windowing)
    lstm_pred_residuals = lstm_results["y_pred_residuals"]
    lstm_dates = lstm_results["test_dates"]

    # Align ARIMA predictions with LSTM dates
    arima_aligned = arima_forecasts.loc[lstm_dates].values
    actual_aligned = arima_test.loc[lstm_dates].values

    # Hybrid = ARIMA + LSTM residual
    hybrid_predictions = arima_aligned + lstm_pred_residuals

    print(f"\n  Aligned test samples: {len(actual_aligned)}")

    # Evaluate all models
    print("\n  ── Model Comparison ──")
    arima_metrics = evaluate_model(actual_aligned, arima_aligned, "ARIMA (standalone)")
    lstm_direct_preds = arima_aligned + lstm_results["y_actual_residuals"]
    hybrid_metrics = evaluate_model(actual_aligned, hybrid_predictions, "Hybrid ARIMA-LSTM")

    # Standalone LSTM (trained directly on AQI, approximate via residuals)
    lstm_resid_metrics = evaluate_model(
        lstm_results["y_actual_residuals"], lstm_pred_residuals, "LSTM (residuals only)"
    )

    # Comparison table
    metrics_df = pd.DataFrame([arima_metrics, lstm_resid_metrics, hybrid_metrics])
    print("\n  ╔════════════════════════════════════════════════════╗")
    print("  ║         MODEL COMPARISON RESULTS                  ║")
    print("  ╠════════════════════════════════════════════════════╣")
    print(metrics_df.to_string(index=False))
    print("  ╚════════════════════════════════════════════════════╝")

    metrics_df.to_csv(os.path.join(RESULTS_DIR, "model_comparison.csv"), index=False)

    # Comparison plot
    plot_comparison(
        actual_aligned,
        {"ARIMA": arima_aligned, "Hybrid ARIMA-LSTM": hybrid_predictions},
        dates=lstm_dates,
        title="Model Comparison — Actual vs Predicted AQI",
        save_path=os.path.join(MODEL_PLOTS_DIR, "model_comparison.png"),
    )

    # Detailed hybrid forecast plot
    setup_plot_style()
    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

    # Panel 1: ARIMA component
    axes[0].plot(lstm_dates, actual_aligned, color="#FFFFFF", linewidth=1, alpha=0.8, label="Actual")
    axes[0].plot(lstm_dates, arima_aligned, color="#FF6B6B", linewidth=1, linestyle="--", label="ARIMA", alpha=0.8)
    axes[0].set_title("ARIMA Component", fontsize=13, fontweight="bold")
    axes[0].set_ylabel("AQI"); axes[0].legend()

    # Panel 2: LSTM residual component
    axes[1].plot(lstm_dates, lstm_results["y_actual_residuals"], color="#FFFFFF",
                 linewidth=1, alpha=0.8, label="Actual Residual")
    axes[1].plot(lstm_dates, lstm_pred_residuals, color="#4ECDC4",
                 linewidth=1, linestyle="--", label="LSTM Predicted", alpha=0.8)
    axes[1].axhline(0, color="#8B949E", linewidth=0.5, linestyle=":")
    axes[1].set_title("LSTM Residual Component", fontsize=13, fontweight="bold")
    axes[1].set_ylabel("Residual"); axes[1].legend()

    # Panel 3: Hybrid combined
    axes[2].plot(lstm_dates, actual_aligned, color="#FFFFFF", linewidth=1.2, alpha=0.9, label="Actual")
    axes[2].plot(lstm_dates, hybrid_predictions, color="#FFEAA7", linewidth=1.2,
                 linestyle="--", label="Hybrid ARIMA-LSTM", alpha=0.9)
    axes[2].fill_between(lstm_dates, actual_aligned, hybrid_predictions, alpha=0.1, color="#FFEAA7")
    axes[2].set_title("Hybrid ARIMA-LSTM Combined Forecast", fontsize=13, fontweight="bold")
    axes[2].set_ylabel("AQI"); axes[2].set_xlabel("Date"); axes[2].legend()

    fig.suptitle("Hybrid ARIMA-LSTM Decomposition", fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_PLOTS_DIR, "hybrid_decomposition.png"))
    plt.close()

    # Improvement summary
    arima_rmse = arima_metrics["RMSE"]
    hybrid_rmse = hybrid_metrics["RMSE"]
    improvement = ((arima_rmse - hybrid_rmse) / arima_rmse) * 100

    print(f"\n  ╔══════════════════════════════════════════╗")
    print(f"  ║  HYBRID MODEL IMPROVEMENT                ║")
    print(f"  ╠══════════════════════════════════════════╣")
    print(f"  ║  ARIMA RMSE:  {arima_rmse:>8.4f}                  ║")
    print(f"  ║  Hybrid RMSE: {hybrid_rmse:>8.4f}                  ║")
    print(f"  ║  Improvement: {improvement:>7.2f}%                  ║")
    print(f"  ╚══════════════════════════════════════════╝")

    return {
        "hybrid_predictions": hybrid_predictions,
        "actual": actual_aligned,
        "arima_preds": arima_aligned,
        "dates": lstm_dates,
        "metrics": {"arima": arima_metrics, "hybrid": hybrid_metrics, "lstm_resid": lstm_resid_metrics},
        "improvement_pct": improvement,
    }


def forecast_future(arima_model, n_days=30):
    """Forecast future AQI values using the ARIMA model."""
    print(f"\n  Forecasting {n_days} days ahead...")
    forecasts, conf_int = arima_model.predict(n_periods=n_days, return_conf_int=True)

    setup_plot_style()
    fig, ax = plt.subplots(figsize=(12, 5))
    days = range(1, n_days + 1)
    ax.plot(days, forecasts, color="#FF6B6B", linewidth=2, label="ARIMA Forecast", marker="o", markersize=3)
    ax.fill_between(days, conf_int[:, 0], conf_int[:, 1], alpha=0.2, color="#FF6B6B", label="95% CI")
    ax.set_title(f"Future {n_days}-Day AQI Forecast", fontsize=14, fontweight="bold")
    ax.set_xlabel("Days Ahead"); ax.set_ylabel("Predicted AQI"); ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_PLOTS_DIR, "future_forecast.png"))
    plt.close()

    forecast_df = pd.DataFrame({
        "Day_Ahead": range(1, n_days + 1),
        "Predicted_AQI": np.round(forecasts, 1),
        "CI_Lower": np.round(conf_int[:, 0], 1),
        "CI_Upper": np.round(conf_int[:, 1], 1),
    })
    forecast_df.to_csv(os.path.join(RESULTS_DIR, "future_forecast.csv"), index=False)
    print(f"  Future forecast saved to results/future_forecast.csv")
    return forecast_df


if __name__ == "__main__":
    print("Run via main.py for full pipeline.")


In [ ]:
hybrid_results = combine_hybrid_forecast(arima_results, lstm_results)

print("\n--- 30-Day Future Forecast ---")
forecast_df = forecast_future(arima_results['model'], n_days=30)
display(forecast_df.head(10))